# 03 — Model Karşılaştırması
LightGBM · XGBoost · CatBoost · Random Forest

Ortak eval framework → leaderboard tablosu → kazanan model seçimi

In [ ]:
# !pip install lightgbm xgboost catboost scikit-learn matplotlib seaborn joblib -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, KFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')

BASE_DIR  = Path('..').resolve()
DATA_DIR  = BASE_DIR / 'data'
MODEL_DIR = BASE_DIR / 'models'
MODEL_DIR.mkdir(exist_ok=True)

COMBO        = ['district_name', 'crop_name', 'variety_name']
TARGET       = 'twso_final'
RANDOM_STATE = 42
print('Hazır.')

## 1. Veri & Feature Yükle

In [ ]:
df = pd.read_parquet(DATA_DIR / 'ml_dataset_gunluk.parquet')

# 02'de kaydedilen feature listesini kullan (varsa), yoksa yeniden üret
feat_path = MODEL_DIR / 'feature_cols.pkl'
if feat_path.exists():
    FEATURE_COLS = joblib.load(feat_path)
else:
    DROP = ['date','district_name','crop_name','variety_name',
            'soil_type','growth_stage','tbase', TARGET]
    FEATURE_COLS = [c for c in df.columns if c not in DROP]
    joblib.dump(FEATURE_COLS, feat_path)

X      = df[FEATURE_COLS].copy()
y      = df[TARGET].copy()
groups = df[COMBO].apply(lambda r: '||'.join(r), axis=1)
dates  = df['date'].copy()

# GroupKFold split (notebook 02 ile aynı mantık)
gkf = GroupKFold(n_splits=5)
CV_SPLITS = list(gkf.split(X, y, groups))
train_idx, test_idx = CV_SPLITS[-1]
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}  |  Features: {len(FEATURE_COLS)}')

## 2. Model Tanımları

In [ ]:
# LightGBM — 02'de kaydedilen en iyi modeli yükle (varsa)
lgbm_path = MODEL_DIR / 'lgbm_yield_final.pkl'
if lgbm_path.exists():
    lgbm_model = joblib.load(lgbm_path)
    print('LightGBM: kaydedilmiş model yüklendi.')
else:
    lgbm_model = lgb.LGBMRegressor(
        n_estimators=1000, learning_rate=0.05, num_leaves=127,
        min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    )
    print('LightGBM: varsayılan parametreler.')

xgb_model = xgb.XGBRegressor(
    n_estimators=1000, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=20,
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
    early_stopping_rounds=50
)

cat_model = CatBoostRegressor(
    iterations=1000, learning_rate=0.05, depth=6,
    random_seed=RANDOM_STATE, verbose=0,
    early_stopping_rounds=50
)

rf_model = RandomForestRegressor(
    n_estimators=300, max_depth=20, min_samples_leaf=10,
    n_jobs=-1, random_state=RANDOM_STATE
)

MODELS = {
    'LightGBM':     lgbm_model,
    'XGBoost':      xgb_model,
    'CatBoost':     cat_model,
    'RandomForest': rf_model,
}
print('Model tanımları hazır.')

## 3. Ortak Eval Framework

In [ ]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, cv_splits, X_full, y_full):
    """Train, CV değerlendirme ve test metrikleri."""
    t0 = time.time()

    # Early stopping destekleyen modeller için eval_set
    fit_kwargs = {}
    if name == 'XGBoost':
        fit_kwargs = {'eval_set': [(X_te, y_te)], 'verbose': False}
    elif name == 'CatBoost':
        fit_kwargs = {'eval_set': (X_te, y_te)}
    elif name == 'LightGBM' and not hasattr(model, 'booster_'):
        fit_kwargs = {
            'eval_set': [(X_te, y_te)],
            'callbacks': [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
        }

    # Eğer LightGBM zaten fit ise tekrar eğitme
    if name == 'LightGBM' and hasattr(model, 'booster_'):
        pass
    else:
        model.fit(X_tr, y_tr, **fit_kwargs)

    train_time = time.time() - t0

    # CV skorları
    cv_rmse = []
    for tr_idx, val_idx in cv_splits:
        preds_val = model.predict(X_full.iloc[val_idx])
        cv_rmse.append(mean_squared_error(y_full.iloc[val_idx], preds_val) ** 0.5)

    # Test metrikleri
    y_pred = model.predict(X_te)
    rmse   = mean_squared_error(y_te, y_pred) ** 0.5
    mae    = mean_absolute_error(y_te, y_pred)
    r2     = r2_score(y_te, y_pred)
    mape   = np.mean(np.abs((y_te - y_pred) / (y_te + 1e-6))) * 100

    result = {
        'Model':        name,
        'RMSE':         round(rmse, 2),
        'MAE':          round(mae, 2),
        'R2':           round(r2, 4),
        'MAPE%':        round(mape, 2),
        'CV_RMSE_mean': round(np.mean(cv_rmse), 2),
        'CV_RMSE_std':  round(np.std(cv_rmse), 2),
        'Train_sec':    round(train_time, 1),
    }
    print(f"{name:15s} → R²={r2:.4f}  RMSE={rmse:.1f}  MAE={mae:.1f}  ({train_time:.0f}s)")
    return result, y_pred


leaderboard_rows = []
all_preds = {}

for name, model in MODELS.items():
    row, preds = evaluate_model(
        name, model,
        X_train, y_train, X_test, y_test,
        CV_SPLITS, X, y
    )
    leaderboard_rows.append(row)
    all_preds[name] = preds

## 4. Leaderboard

In [ ]:
leaderboard = pd.DataFrame(leaderboard_rows).sort_values('R2', ascending=False).reset_index(drop=True)
leaderboard.index += 1
leaderboard.index.name = 'Rank'
print('\n══ LEADERBOARD ══')
print(leaderboard.to_string())

WINNER = leaderboard.iloc[0]['Model']
print(f'\n🏆 Kazanan: {WINNER}')

In [ ]:
# Leaderboard görsel
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics = [('R2', True, 'steelblue'), ('RMSE', False, 'darkorange'), ('MAPE%', False, 'mediumpurple')]

for ax, (metric, higher_better, color) in zip(axes, metrics):
    data = leaderboard.sort_values(metric, ascending=not higher_better)
    bars = ax.barh(data['Model'], data[metric], color=color, edgecolor='white')
    ax.bar_label(bars, fmt='%.3f' if metric == 'R2' else '%.1f', padding=3)
    ax.set_title(metric)
    ax.set_xlabel(metric)

plt.suptitle('Model Karşılaştırması', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Tahmin Karşılaştırması (Scatter)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, (name, preds) in zip(axes.flat, all_preds.items()):
    r2c = r2_score(y_test, preds)
    ax.scatter(y_test, preds, alpha=0.25, s=6, color='steelblue')
    lims = [y_test.min(), y_test.max()]
    ax.plot(lims, lims, 'r--', lw=1.5)
    ax.set_title(f'{name}  (R²={r2c:.3f})')
    ax.set_xlabel('Gerçek')
    ax.set_ylabel('Tahmin')

plt.suptitle('Tüm Modeller — Gerçek vs Tahmin', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Kazanan Modeli Kaydet

In [ ]:
winner_model = MODELS[WINNER]
winner_path  = MODEL_DIR / 'best_model.pkl'
joblib.dump(winner_model, winner_path)

leaderboard.to_csv(MODEL_DIR / 'leaderboard.csv', index=True)

print(f'Kazanan model kaydedildi : {winner_path}')
print(f'Leaderboard kaydedildi   : {MODEL_DIR}/leaderboard.csv')
print(f'\nSonraki adım: 04_inference_pipeline.py')